# 01 — Road-Network Graph Construction

This notebook walks through how we represent a city's road network as a **directed weighted graph** and prepares the adjacency / diffusion matrices used by DCRNN.

**Outputs**
- `data/metr-la/adj_mx.npy` — dense adjacency matrix
- `data/metr-la/node_coords.csv` — sensor GPS coordinates
- `data/metr-la/graph.gpickle` — NetworkX DiGraph
- `data/metr-la/graph_map.html` — interactive Folium map

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pickle

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

from config import DATA_DIR, cfg
from src.utils import build_support, compute_diffusion_matrices

sns.set_theme(style='darkgrid')
print('Imports OK')

## 1. Load adjacency pickle

In [ ]:
adj_pkl = DATA_DIR / 'metr-la' / 'adj_mx.pkl'

with open(adj_pkl, 'rb') as f:
    sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(f, encoding='latin1')

adj_mx = adj_mx.astype(np.float32)
N      = adj_mx.shape[0]

print(f'Nodes: {N}')
print(f'Edges: {(adj_mx > 0).sum()}')
print(f'Density: {(adj_mx > 0).mean():.3%}')

## 2. Visualise adjacency matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap (first 50 nodes for clarity)
sns.heatmap(adj_mx[:50, :50], ax=axes[0], cmap='viridis', cbar=True)
axes[0].set_title('Adjacency Matrix (first 50 × 50 nodes)')

# Edge-weight distribution
weights = adj_mx[adj_mx > 0].flatten()
axes[1].hist(weights, bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Edge weight')
axes[1].set_ylabel('Count')
axes[1].set_title('Edge Weight Distribution')

plt.tight_layout()
plt.show()

## 3. Build diffusion support matrices

In [ ]:
supports = build_support(adj_mx, filter_type='dual_random_walk', max_diffusion_step=2)

print(f'Number of support matrices: {len(supports)}')
for i, s in enumerate(supports):
    print(f'  support[{i}]: shape={s.shape}, sparsity={np.mean(s == 0):.3%}')

## 4. NetworkX graph statistics

In [ ]:
G = nx.from_numpy_array(adj_mx, create_using=nx.DiGraph)

print('Graph statistics')
print(f'  Nodes:               {G.number_of_nodes()}')
print(f'  Edges:               {G.number_of_edges()}')
print(f'  Avg out-degree:      {np.mean([d for _, d in G.out_degree()]):.2f}')
print(f'  Avg in-degree:       {np.mean([d for _, d in G.in_degree()]):.2f}')
print(f'  Is weakly connected: {nx.is_weakly_connected(G)}')

# Degree distribution
degrees = [d for _, d in G.out_degree()]
plt.figure(figsize=(8, 4))
plt.hist(degrees, bins=30, color='salmon', edgecolor='white')
plt.xlabel('Out-degree')
plt.ylabel('Number of nodes')
plt.title('Out-degree Distribution — METR-LA Graph')
plt.tight_layout()
plt.show()

## 5. Generate interactive map (requires folium)

In [ ]:
from src.build_graph import build_metr_la

build_metr_la(DATA_DIR / 'metr-la')

map_path = DATA_DIR / 'metr-la' / 'graph_map.html'
if map_path.exists():
    from IPython.display import IFrame
    display(IFrame(str(map_path), width='100%', height='500px'))